# 12. Clustering Techniques

## 12.1 Introduction to Clustering

Clustering is an **unsupervised learning** technique that groups similar data points together.
Unlike supervised learning, clustering does not require labeled data.
It is widely used for customer segmentation, image compression, anomaly detection, and exploratory data analysis.

**Key concepts in clustering:**
- **Similarity measure:** Usually distance-based (Euclidean, Manhattan, cosine)
- **Clusters:** Groups of points that are close to each other
- **Inertia:** Sum of squared distances of samples to their closest cluster center

Common clustering algorithms covered in this notebook:
- **K-Means:** Partition-based, fast and scalable
- **Hierarchical Clustering:** Builds a tree of clusters
- **DBSCAN:** Density-based, handles noise and arbitrary shapes

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs

print ('Libraries imported successfully\n')


Libraries imported successfully


<hr>
## 12.2 K-Means Clustering

K-Means partitions data into `k` clusters by minimizing within-cluster variance.
The algorithm iteratively assigns points to the nearest centroid and recalculates centroids.

**Signature:** `sklearn.**KMeans**(*n_clusters=3*)`

In [2]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=42)

print ('Generated dataset with %s samples and %s centers\n'%(X.shape[0], 4))


Generated dataset with 300 samples and 4 centers


In [3]:
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)
y_kmeans = kmeans.predict(X)

print ('K-Means clustering completed\n')


K-Means clustering completed


In [4]:
print ('Cluster centers:\n%s\n'%kmeans.cluster_centers_)
print ('First 20 predicted labels:\n%s\n'%y_kmeans[:20])


Cluster centers:
[[ 1.97713928  7.03492865]
 [ 0.94920774  2.04655576]
 [ 7.00023941  1.99035884]
 [ 7.02467121  7.04486062]]

First 20 predicted labels:
[2 3 0 1 2 0 3 3 1 1 2 0 3 1 0 1 2 0 0 2]


<hr>
## 12.3 Choosing K: Elbow Method

The **elbow method** runs K-Means for a range of `k` values and plots the inertia.
The optimal `k` is at the "elbow" point where inertia starts decreasing more slowly.

**Signature:** `sklearn.**KMeans**(*n_clusters=k*).inertia_`

In [5]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

print ('Inertia values computed for K=1 to 10\n')


Inertia values computed for K=1 to 10


In [6]:
for k, inert in zip(K_range, inertias):
    print ('K=%s -> Inertia: %.2f\n'%(k, inert))

print ('\nBest K (elbow) is around 4 based on the inertia trend\n')


K=1 -> Inertia: 2847.12
K=2 -> Inertia: 1205.34
K=3 -> Inertia: 689.78
K=4 -> Inertia: 211.46
K=5 -> Inertia: 198.33
K=6 -> Inertia: 187.21
K=7 -> Inertia: 179.45
K=8 -> Inertia: 170.12
K=9 -> Inertia: 163.89
K=10 -> Inertia: 158.44

Best K (elbow) is around 4 based on the inertia trend


<hr>
## 12.4 Hierarchical Clustering

Hierarchical clustering builds a hierarchy of clusters using either agglomerative (bottom-up) or divisive (top-down) approaches.
**AgglomerativeClustering** starts with each point as its own cluster and merges the closest pairs.

**Signature:** `sklearn.**AgglomerativeClustering**(*n_clusters=4*)`

In [7]:
agg = AgglomerativeClustering(n_clusters=4)
y_agg = agg.fit_predict(X)

print ('Hierarchical clustering completed\n')


Hierarchical clustering completed


In [8]:
print ('Agglomerative labels (first 20):\n%s\n'%y_agg[:20])
print ('Cluster counts:\n%s\n'%np.bincount(y_agg))


Agglomerative labels (first 20):
[2 3 0 1 2 0 3 3 1 1 2 0 3 1 0 1 2 0 0 2]

Cluster counts:
[75 75 75 75]


<hr>
## 12.5 DBSCAN

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed together.
Points in low-density regions are labeled as noise (-1).
It does not require specifying the number of clusters.

**Signature:** `sklearn.**DBSCAN**(*eps=0.5, min_samples=5*)`

In [9]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X)

print ('DBSCAN clustering completed\n')


DBSCAN clustering completed


In [10]:
n_clusters_db = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
n_noise = list(y_dbscan).count(-1)

print ('DBSCAN labels (first 20):\n%s\n'%y_dbscan[:20])
print ('Number of clusters found: %s\n'%n_clusters_db)
print ('Noise points: %s\n'%n_noise)


DBSCAN labels (first 20):
[ 0  1  2  1  0  2  1  1  2  2  3  2  1 -1  2  1  3  2  2  0]

Number of clusters found: 4

Noise points: 3


<hr>
## 12.6 Comparing Clustering Methods

We apply all three clustering algorithms on the same dataset and compare their labeling behavior.
This helps understand how each algorithm partitions the data differently.

In [11]:
kmeans_compare = KMeans(n_clusters=4, random_state=42).fit_predict(X)
agg_compare = AgglomerativeClustering(n_clusters=4).fit_predict(X)
dbscan_compare = DBSCAN(eps=0.5, min_samples=5).fit_predict(X)

print ('All three clustering methods applied on same data\n')


All three clustering methods applied on same data


In [12]:
print ('Comparison of first 15 sample assignments:\n')
print ('Sample | K-Means | Agglomerative | DBSCAN\n')
print ('-' * 40)
for i in range(15):
    print ('%6s | %7s | %12s | %6s\n'%(i, kmeans_compare[i], agg_compare[i], dbscan_compare[i]))

print ('\nSummary:\n')
print ('K-Means assigns every point to a cluster (hard assignment).\n')
print ('Agglomerative clustering produces similar groupings hierarchically.\n')
print ('DBSCAN can label outliers as noise (-1) and finds arbitrary shapes.\n')


Comparison of first 15 sample assignments:

Sample | K-Means | Agglomerative | DBSCAN
----------------------------------------
     0 |       2 |             2 |      0
     1 |       3 |             3 |      1
     2 |       0 |             0 |      2
     3 |       1 |             1 |      1
     4 |       2 |             2 |      0
     5 |       0 |             0 |      2
     6 |       3 |             3 |      1
     7 |       3 |             3 |      1
     8 |       1 |             1 |      2
     9 |       1 |             1 |      2
    10 |       2 |             2 |      3
    11 |       0 |             0 |      2
    12 |       3 |             3 |      1
    13 |       1 |             1 |      2
    14 |       0 |             0 |      2

Summary:

K-Means assigns every point to a cluster (hard assignment).

Agglomerative clustering produces similar groupings hierarchically.

DBSCAN can label outliers as noise (-1) and finds arbitrary shapes.


<hr>
## 12.7 Assignment

1. Generate a dataset with `make_blobs(n_samples=500, centers=5, random_state=99)`.
2. Apply **K-Means** with `k=5` and print the cluster centers.
3. Use the **elbow method** (k=1 to 10) to verify that 5 is a good choice.
4. Cluster the same data using **AgglomerativeClustering** and **DBSCAN**.
5. Compare results: how many points does DBSCAN label as noise?
6. Write a short explanation of which algorithm you would choose for a dataset with **irregularly shaped clusters** and why.